This is a notebook to compare the source STL shapes that we'll use to make the 3D printed objects to the scanned point clouds from the Intel Real Sense cameras. 

In [2]:
# imports and path configuration
import sys
import os
import open3d as o3d
import numpy as np
import glob
import pandas as pd

# Add the 'Experimental' directory to the system path so we can import from '3d scan'
# Note: Since '3d scan' has a space, we use a slightly different approach to import it
sys.path.append(os.path.abspath("Experimental"))

# We use importlib to handle the folder name with a space
import importlib.util
module_name = "three_d_scan" # We treat it as a module
file_path = os.path.join("3d scan", "process_3d_files.py")

# Load the module manually to bypass the space in the folder name
spec = importlib.util.spec_from_file_location(module_name, file_path)
p3f = importlib.util.module_from_spec(spec)
spec.loader.exec_module(p3f)

# import script used to generate the actual
gen_file_path = os.path.abspath(r"D:\Projects\3D-textures\Generator\Generator.py")

# We use importlib to handle the import, same as we did for the 3d scan folder
spec = importlib.util.spec_from_file_location("Generator", gen_file_path)
gen_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gen_module)

#setup paths for use
SCANS_LOC = r"D:\Projects\3D-textures\processed_models"

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
#debugging alignment
#Do complete loading and comparison process
# load ref
ref_obj = gen_module.surface_to_stl(gen_module.x, gen_module.y, gen_module.z1, filename=None)
# load scan
ply_path1 = r"D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply"
scan_obj = p3f.process_ply(ply_path1, rm_floor=True)

# --- Visualisation 1: Raw comparison ---
# Paint them different colors so you can see the overlap/offset clearly
ref_obj.paint_uniform_color([1, 0, 0])  # Red
scan_obj.paint_uniform_color([0, 1, 0])  # Green
print("Showing Raw Comparison (Red: STL, Green: Scan)...")
o3d.visualization.draw_geometries([ref_obj, scan_obj])

# --- Next scale the scan according to the ref ---
# We use your helper function to fix the size difference
scaled_scan_obj = p3f.scale_pcd_to_reference(ref_obj, scan_obj)

# --- Visualisation 2: After scaling ---
# The green cloud should now be roughly the same size as the red cloud
scaled_scan_obj.paint_uniform_color([0, 1, 0]) 
print("Showing Scaled Comparison...")
o3d.visualization.draw_geometries([ref_obj, scaled_scan_obj])

# --- Next perform alignment ---
# We set rm_floor=False and scale=False because those steps are already complete
# align_ply_objects returns (ref_obj, aligned_scan)
ref_obj, aligned_scan = p3f.align_ply_objects(ref_obj, scaled_scan_obj, rm_floor=False, scale=False)

# --- Visualisation 3: Final Alignment ---
# The green cloud should now be "snapped" onto the red cloud
aligned_scan.paint_uniform_color([0, 1, 0])
print("Showing Final Alignment...")
o3d.visualization.draw_geometries([ref_obj, aligned_scan])

# Final step: Calculate metrics
avg, std = p3f.calc(ref_obj, aligned_scan)
print(f"Final Metrics -> Average Deviation: {avg:.4f}, STD: {std:.4f}")

Returned PointCloud (no file saved)
Processed D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply (no file saved)
Showing Raw Comparison (Red: STL, Green: Scan)...
Detected Scale Factor: 937.5310x
Showing Scaled Comparison...
Showing Final Alignment...
Final Metrics -> Average Deviation: 8.1261, STD: 5.9121


In [ ]:
# making version 2 of the align function
# This function is from the "align_ply_objects" function from the original file process_3d_files.py, but I'm editing it to fix the alignment. 
# this version has an additional alignment step
def align_stlpcd_pcd(stl_pcd, pcd, rm_floor=False, scale=False, use_global_reg=True):
    """
    Aligns two existing PointCloud objects using a Robust Multi-Stage ICP.
    
    Since FPFH Global Registration is unavailable in this Open3D build,
    we use a 'Super Loose' initial gate to handle rotation, followed by 
    standard multi-stage ICP for sub-millimeter refinement.
    
    Args:
        stl_pcd (o3d.geometry.PointCloud): Point cloud from stl.
        pcd (o3d.geometry.PointCloud): Point cloud from scan.
        rm_floor (bool): Whether to run the floor removal logic on the scan pointcloud object.
            Defaults to False.
        scale (bool): Whether to scale pcd to the scale of stl_pcd.
        use_global_reg (bool): If True, uses a wide initial gate to solve rotation.
            If False, uses a standard tight gate for refinement.
            Defaults to True.

    Returns:
        tuple: (stl_pcd, pcd) where pcd is aligned to stl_pcd, or None if failed.
    """
    try:
        # Check if inputs are valid point clouds
        if stl_pcd is None or pcd is None:
            print("Error: One or both input objects are None.")
            return None
        
        if len(stl_pcd.points) == 0 or len(pcd.points) == 0:
            print(f"Error: One of the objects is empty. STL: {len(stl_pcd.points)}, PCD: {len(pcd.points)}")
            return None

        # --- CENTERING STEP ---
        # This handles the translation (moving objects to the same spot)
        stl_center = stl_pcd.get_center()
        pcd_center = pcd.get_center()
        stl_pcd.translate(-stl_center)
        pcd.translate(-pcd_center)

        # Clean floor elements
        if rm_floor:
            pcd = p3f.remove_floor(pcd)

        # Scale pcd to the scale of stl_pcd
        if scale:
            pcd = p3f.scale_pcd_to_reference(stl_pcd, pcd)

        # --- MULTI-STAGE ICP REFINEMENT ---
        voxel_size_refine = 0.005

        if use_global_reg:
            # "Super Loose" mode: Use a very wide gate to "grab" the rotation.
            # 0.150 (15cm) gives the algorithm a massive reach to find the correct
            # orientation even if the scan is rotated significantly.
            stages = [
                {"voxel": voxel_size_refine * 2, "thresh": 0.150},  # Super Loose (Wide Gate)
                {"voxel": voxel_size_refine * 1, "thresh": 0.015},  # Medium
                {"voxel": voxel_size_refine * 0.5, "thresh": 0.004} # Tight
            ]
            print("Running Robust Alignment (Wide Gate Mode)...")
        else:
            # "Standard" mode: Use for when the object is already mostly aligned.
            stages = [
                {"voxel": voxel_size_refine * 2, "thresh": 0.040},  # Loose
                {"voxel": voxel_size_refine * 1, "thresh": 0.015},  # Medium
                {"voxel": voxel_size_refine * 0.5, "thresh": 0.004} # Tight
            ]
            print("Running Standard Refinement...")

        current_transformation = np.identity(4)

        for i, stage in enumerate(stages):
            s_stl = stl_pcd.voxel_down_sample(stage["voxel"])
            s_pcd = pcd.voxel_down_sample(stage["voxel"])
            
            result = o3d.pipelines.registration.registration_icp(
                source=s_pcd,
                target=s_stl,
                max_correspondence_distance=stage["thresh"],
                init=current_transformation,
                estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
            )
            current_transformation = result.transformation
            print(f"ICP Stage {i+1} complete. Fitness: {result.fitness:.4f}")

        # Apply final transformation to full-resolution cloud
        pcd.transform(current_transformation)

        # Colorize
        stl_pcd.paint_uniform_color([1, 0, 0])  # Red
        pcd.paint_uniform_color([0, 1, 0])  # Green
        
        return stl_pcd, pcd

    except Exception as e:
        print(f"Alignment execution failed: {e}")
        import traceback
        traceback.print_exc()
        return None

In [22]:
# load ref
ref_obj = gen_module.surface_to_stl(gen_module.x, gen_module.y, gen_module.z1, filename=None)
# load scan
ply_path1 = r"D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply"
scan_obj = p3f.process_ply(ply_path1, rm_floor=True)

ref_obj_a, scan_obj_a = align_stlpcd_pcd(ref_obj, scan_obj, scale=True)
o3d.visualization.draw_geometries([ref_obj_a, scan_obj_a])
avg, std = p3f.calc(ref_obj_a, scan_obj_a)
print(f"Final Metrics -> Average Deviation: {avg:.4f}, STD: {std:.4f}")

Returned PointCloud (no file saved)
Processed D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply (no file saved)
Detected Scale Factor: 937.4590x
Running Robust Alignment (Wide Gate Mode)...
ICP Stage 1 complete. Fitness: 0.2893
ICP Stage 2 complete. Fitness: 0.0026
ICP Stage 3 complete. Fitness: 0.0006
Final Metrics -> Average Deviation: 0.4659, STD: 0.4057


Now that we have a reliable way to align and compare point clouds from the ST and the SCAMPLYs, we need to compare them in large quantities.
First by confirming what the z values correspond to.

In [65]:
# ==========================
# Batch Comparison for All Folders
# ==========================
all_ply_files = []

# Iterate through all items in the SCANS_LOC directory
for entry in os.scandir(SCANS_LOC):
    if entry.is_dir():
        # Find all .ply files in this subdirectory (and sub-sub-directories if they exist)
        # recursive=True allows it to find files in "BAMBU STANDARDS" or similar nested paths
        files = glob.glob(os.path.join(entry.path, "**", "*.ply"), recursive=True)
        all_ply_files.extend(files)

# Sort the files alphabetically to maintain a consistent order
all_ply_files.sort()

results_dicts = []

if not all_ply_files:
    print(f"No .ply files found in {SCANS_LOC}")
else:
    print(f"Found {len(all_ply_files)} files across all folders. Starting automated comparison...")
    
    for i, file_path in enumerate(all_ply_files):
        # Extract filename and path info
        filename_only = os.path.splitext(os.path.basename(file_path))[0]
        path_parts = file_path.split(os.sep)
        
        # Determine Printer and Standards
        # Logic: Find the folder name immediately following 'processed_models'
        folder_name = os.path.basename(os.path.dirname(file_path))
        #check whether standard is in the folder name
        is_standard = "STANDARDS" in folder_name
        #set printer name. STANDARDS needs removing if its from a standards folder
        if is_standard:
            printer = folder_name[:folder_name.find("STANDARDS")-1]
        else:
            printer = folder_name

        # Determine the relative path for the 'filename' column 
        # (From processed_models/ onwards)
        parent_dir = os.path.dirname(SCANS_LOC)
        rel_path = os.path.relpath(file_path, parent_dir)

        try:
            # Parsing filename "B1T2"
            # stl_texture is the 2nd character (index 1)
            # test_no is the 4th character (index 3)
            stl_texture = int(filename_only[1])
            test_no = int(filename_only[3])
            
            # Dynamically get the correct z_data (z1, z2, etc.) from the generator
            z_data = getattr(gen_module, f"z{stl_texture}")
            
            print(f"\n--- Processing [{i+1}/{len(all_ply_files)}]: {filename_only} ---")
            print(f"Folder: {folder_name} | Texture: {stl_texture} | Test: {test_no} | Standard: {is_standard}")

            # 1. Generate the matching reference STL
            ref_obj = gen_module.surface_to_stl(gen_module.x, gen_module.y, z_data, filename=None)
            
            # 2. Process the scan (remove floor, etc.)
            scan_obj = p3f.process_ply(file_path, rm_floor=True)

            if ref_obj is not None and scan_obj is not None:
                # 3. Perform Alignment
                ref_obj_a, scan_obj_a = align_stlpcd_pcd(ref_obj, scan_obj, scale=True)
                
                # 4. Visualize the result
                # Now includes the Folder Display (e.g., "BAMBU STANDARDS") and the filename
                #o3d.visualization.draw_geometries([ref_obj_a, scan_obj_a], 
                                                   #window_name=f"Align: {folder_display} | {filename_only}")
                
                # 5. Calculate and print metrics
                avg, std = p3f.calc(ref_obj_a, scan_obj_a)

                # Automatically set 'aligned' to False if it exceeds the threshold
                # Define your alignment threshold based on distance cal
                threshold = 1.2
                is_aligned = True if avg < threshold else False
                
                # Store results in list
                results_dicts.append({
                    "filename": rel_path,
                    "printer": printer,
                    "standards": is_standard,
                    "stl_texture": stl_texture,
                    "test_no": test_no,
                    "avg_dist": avg,
                    "std_dist": std,
                    "aligned": is_aligned
                })
                
                print(f"Final Metrics -> Average Deviation: {avg:.4f}, STD: {std:.4f}")
            else:
                print(f"Warning: Failed to load objects for {filename_only}")

        except (IndexError, ValueError) as e:
            print(f"Skipping {filename_only}: Filename does not follow convention (e.g. B1T2). Error: {e}")
        except AttributeError:
            print(f"Skipping {filename_only}: No 'z' data found for index {stl_texture} in Generator.")
        except Exception as e:
            print(f"Unexpected error processing {filename_only}: {e}")

# Create DataFrame and display
results_df = pd.DataFrame(results_dicts)
print("\n--- Final Comparison Summary ---")
#print(results_df)

# Optional: Save to CSV
# results_df.to_csv("scan_comparison_results.csv", index=False)

results_df.head()

Found 190 files across all folders. Starting automated comparison...

--- Processing [1/190]: B1T1 ---
Folder: BAMBU STANDARDS | Texture: 1 | Test: 1 | Standard: True
Returned PointCloud (no file saved)
Processed D:\Projects\3D-textures\processed_models\BAMBU STANDARDS\B1T1.ply (no file saved)
Detected Scale Factor: 919.4493x
Running Robust Alignment (Wide Gate Mode)...
ICP Stage 1 complete. Fitness: 0.2740
ICP Stage 2 complete. Fitness: 0.0027
ICP Stage 3 complete. Fitness: 0.0002
Final Metrics -> Average Deviation: 0.4795, STD: 0.4060

--- Processing [2/190]: B1T2 ---
Folder: BAMBU STANDARDS | Texture: 1 | Test: 2 | Standard: True
Returned PointCloud (no file saved)
Processed D:\Projects\3D-textures\processed_models\BAMBU STANDARDS\B1T2.ply (no file saved)
Detected Scale Factor: 920.4025x
Running Robust Alignment (Wide Gate Mode)...
ICP Stage 1 complete. Fitness: 0.2646
ICP Stage 2 complete. Fitness: 0.0018
ICP Stage 3 complete. Fitness: 0.0000
Final Metrics -> Average Deviation: 0.4

,filename,printer,standards,stl_texture,test_no,avg_dist,std_dist,aligned
0,processed_models\BAMBU STANDARDS\B1T1.ply,BAMBU,True,1,1,0.479479,0.406014,True
1,processed_models\BAMBU STANDARDS\B1T2.ply,BAMBU,True,1,2,0.473799,0.386038,True
2,processed_models\BAMBU STANDARDS\B1T3.ply,BAMBU,True,1,3,0.491450,0.420577,True
3,processed_models\BAMBU STANDARDS\B1T4.ply,BAMBU,True,1,4,0.486166,0.414465,True
4,processed_models\BAMBU STANDARDS\B1T5.ply,BAMBU,True,1,5,0.402532,0.341339,True


In [68]:
results_df = pd.DataFrame(results_dicts)
print(results_df.tail().to_string())
# Optional: Save to CSV
results_df.to_csv("scan_comp_master_results.csv", index=False)

                            filename printer  standards  stl_texture  test_no   avg_dist  std_dist  aligned
185  processed_models\RESIN\R6T1.ply   RESIN      False            6        1  11.440968  7.838545    False
186  processed_models\RESIN\R6T2.ply   RESIN      False            6        2   0.711651  0.445554     True
187  processed_models\RESIN\R6T3.ply   RESIN      False            6        3   0.785713  0.475554     True
188  processed_models\RESIN\R6T4.ply   RESIN      False            6        4   0.753800  0.581768     True
189  processed_models\RESIN\R6T5.ply   RESIN      False            6        5   0.748504  0.443160     True
